In [3]:
import pandas as pd
import faiss
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer
import scipy.sparse
import pickle
import numpy as np
from dotenv import load_dotenv

In [20]:
df = pd.read_csv('data.csv')
df = df.fillna("")

In [21]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [22]:
user_tags = ["Comedy & Performance", "Music", "Food & Drink"]
df["combined_categories"] = df["category_1"] + " | " + df["category_2"] + " | " + df["category_3"]

In [23]:
event_embeddings = model.encode(df["combined_categories"].tolist(), convert_to_numpy=True)

In [24]:
dimension = event_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(event_embeddings)

In [26]:
faiss.write_index(index, "events.index")
with open("events_metadata.pkl", "wb") as f:
    pickle.dump(df, f)

In [27]:
loaded_index = faiss.read_index("events.index")
with open("events_metadata.pkl", "rb") as f:
    loaded_events = pickle.load(f)

In [30]:
user_tags = ["Comedy & Performance", "Food & Drink", "Music"]
user_embedding = model.encode([" | ".join(user_tags)], convert_to_numpy=True)
distances, indices = loaded_index.search(user_embedding, k=10)

results = []
for dist, idx in zip(distances[0], indices[0]):
    results.append({
        "event_id": df.iloc[idx]["event_id"],
        "similarity_score": float(dist)  # distance from FAISS (smaller = more similar)
    })

In [31]:
print(results)

[{'event_id': np.int64(1028534082127), 'similarity_score': 0.11116497218608856}, {'event_id': np.int64(1656832664099), 'similarity_score': 0.2577621638774872}, {'event_id': np.int64(1612316314499), 'similarity_score': 0.2972962260246277}, {'event_id': np.int64(1515971133469), 'similarity_score': 0.3502162992954254}, {'event_id': np.int64(1657920989309), 'similarity_score': 0.3502162992954254}, {'event_id': np.int64(1571899707339), 'similarity_score': 0.4224414527416229}, {'event_id': np.int64(1554921394749), 'similarity_score': 0.4278407692909241}, {'event_id': np.int64(567396166207), 'similarity_score': 0.44280779361724854}, {'event_id': np.int64(1559643669199), 'similarity_score': 0.44280779361724854}, {'event_id': np.int64(1698615608009), 'similarity_score': 0.44280779361724854}]
